# MeshPort · Colab A100

Run all cells, then paste the complete URL printed by the final cell into the localhost app. Upload one image or up to six views of the same object in localhost. Generation runs here; preview and textured FBX conversion run locally.

Select an A100 GPU before running. First setup and downloads can take several minutes. The tunnel requires a random session key included in the URL fragment. Treat the complete URL as private; notebook outputs may contain that key. Clear outputs before sharing the notebook. Keep Colab connected.

Uses TRELLIS-image-large with seed 42, 16 sampling steps and 2048 textures. For multiple inputs, TRELLIS blends each view at every geometry and appearance step. This requires more GPU time as the number of views grows. Multi-image conditioning is experimental.


In [ ]:
# @title 0. Verify runtime before setup
import sys, subprocess, pathlib
print("Colab kernel Python:", sys.version)
for cmd in [["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], ["nvcc", "--version"]]:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout, result.stderr)
print("Existing TRELLIS:", pathlib.Path("/content/TRELLIS").exists())
print("Input images:", [str(p) for p in pathlib.Path("/content").glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}])


In [ ]:
# @title 1. Install an isolated TRELLIS environment (rerunnable)
import os, pathlib, subprocess, textwrap
setup_script = r'''
set -euo pipefail
cd /content
export MAMBA_ROOT_PREFIX=/content/mamba
if [ ! -x /content/bin/micromamba ]; then
  curl -fLs https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
  tar -xjf /content/micromamba.tar.bz2 -C /content bin/micromamba
fi
if [ ! -x /content/trellis-env/bin/python ]; then
  /content/bin/micromamba create -y -p /content/trellis-env -c conda-forge -c nvidia python=3.10 pip cuda-nvcc=12.1 cuda-cudart-dev=12.1 cuda-cccl=12.1 gcc_linux-64=12 gxx_linux-64=12
fi
export PATH=/content/trellis-env/bin:$PATH
export CUDA_HOME=/content/trellis-env
export CC=/content/trellis-env/bin/x86_64-conda-linux-gnu-cc
export CXX=/content/trellis-env/bin/x86_64-conda-linux-gnu-c++
export TORCH_CUDA_ARCH_LIST=8.0 MAX_JOBS=4
export ATTN_BACKEND=xformers SPCONV_ALGO=native
if [ ! -d /content/TRELLIS/.git ]; then
  git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
fi
cd /content/TRELLIS
git rev-parse HEAD
printf 'numpy==1.26.4
scipy==1.14.1
torch==2.4.0
torchvision==0.19.0
xformers==0.0.27.post2
pymeshfix==0.17.0
pyvista==0.44.2
vtk==9.3.1
transformers==4.46.3
huggingface-hub==0.25.2
fastapi==0.115.6
pydantic==2.10.6
starlette==0.41.3
' > /content/trellis-constraints.txt
export PIP_CONSTRAINT=/content/trellis-constraints.txt
python -m pip install --disable-pip-version-check 'setuptools<70' wheel ninja packaging
python -m pip install torch==2.4.0 torchvision==0.19.0 xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
python -m pip install numpy==1.26.4 scipy==1.14.1 pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless rembg==2.0.61 onnxruntime trimesh open3d xatlas pyvista==0.44.2 pymeshfix==0.17.0 igraph transformers==4.46.3 huggingface-hub==0.25.2 spconv-cu120 gradio==4.44.1 fastapi==0.115.6 pydantic==2.10.6 starlette==0.41.3
python -m pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
python -m pip install kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html
export CPATH=/content/trellis-env/targets/x86_64-linux/include
for cuda_include in /content/trellis-env/lib/python3.10/site-packages/nvidia/*/include; do
  export CPATH="$CPATH:$cuda_include"
done
export LIBRARY_PATH=/content/trellis-env/targets/x86_64-linux/lib
mkdir -p /content/trellis-extensions
if [ ! -d /content/trellis-extensions/nvdiffrast/.git ]; then
  git clone --branch v0.3.3 https://github.com/NVlabs/nvdiffrast.git /content/trellis-extensions/nvdiffrast
fi
if [ ! -d /content/trellis-extensions/mip-splatting/.git ]; then
  git clone --recursive https://github.com/autonomousvision/mip-splatting.git /content/trellis-extensions/mip-splatting
fi
if ! python -c 'import nvdiffrast' >/dev/null 2>&1; then
  python -m pip install --no-build-isolation /content/trellis-extensions/nvdiffrast
fi
if ! python -c 'import diff_gaussian_rasterization' >/dev/null 2>&1; then
  python -m pip install --no-build-isolation /content/trellis-extensions/mip-splatting/submodules/diff-gaussian-rasterization
fi
python -m pip install numba
python -m pip check
python -u -c "import torch,numpy,scipy,kaolin,nvdiffrast.torch,diff_gaussian_rasterization; from trellis.pipelines import TrellisImageTo3DPipeline; from trellis.utils import postprocessing_utils; assert torch.cuda.is_available(); nvdiffrast.torch.RasterizeCudaContext(); print('VERIFIED',torch.__version__,numpy.__version__,scipy.__version__,torch.cuda.get_device_name(0))"
printf '
TRELLIS dependency imports passed. Run Cell 2.
'
'''
pathlib.Path('/content/trellis-setup.sh').write_text(textwrap.dedent(setup_script))
with open('/content/trellis-setup.log', 'a') as log:
    proc = subprocess.Popen(['bash', '/content/trellis-setup.sh'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        print(line, end='')
    rc = proc.wait()
if rc:
    raise RuntimeError(f'Setup stopped with exit code {rc}; see /content/trellis-setup.log. Completed packages are reusable.')


In [ ]:
# @title 2. Start the image-to-3D service
import os, pathlib, subprocess, secrets, time, urllib.request
service_source = '"""Authenticated Colab service, embedded into the notebook by build_notebook.py."""\nimport os, io, base64, json, uuid, time, threading, hashlib\nfrom pathlib import Path\nos.environ[\'ATTN_BACKEND\']=\'xformers\'\nos.environ[\'SPCONV_ALGO\']=\'native\'\nimport torch\nfrom PIL import Image, ImageOps\nfrom fastapi import FastAPI, Request, HTTPException\nfrom fastapi.responses import FileResponse, JSONResponse\nfrom trellis.pipelines import TrellisImageTo3DPipeline\nfrom trellis.utils import postprocessing_utils\nfrom photo_finish import refine, unlit\n\nTOKEN=os.environ[\'STUDIO_KEY\']\nROOT=Path(\'/content/trellis-results\'); ROOT.mkdir(exist_ok=True)\npipeline=TrellisImageTo3DPipeline.from_pretrained(\'microsoft/TRELLIS-image-large\')\npipeline.cuda()\napp=FastAPI(docs_url=None,redoc_url=None,openapi_url=None)\njobs={}; lock=threading.Lock()\nImage.MAX_IMAGE_PIXELS=16_000_000\nMULTI_VIEW_MODE=\'multidiffusion\'\nSIMPLIFY_RATIO = float(os.environ.get(\'MESHPORT_SIMPLIFY_RATIO\', \'0.50\'))\nif not 0 <= SIMPLIFY_RATIO < 1:\n    raise ValueError(\'MESHPORT_SIMPLIFY_RATIO must be in [0, 1).\')\n\n@app.middleware(\'http\')\nasync def auth(request,call_next):\n    import secrets\n    if not secrets.compare_digest(request.headers.get(\'authorization\',\'\'),\'Bearer \'+TOKEN):\n        return JSONResponse({\'detail\':\'Invalid connection key\'},401)\n    return await call_next(request)\n\n@app.get(\'/health\')\ndef health():\n    return {\'protocol\':\'form-studio-v1\',\'ready\':True,\'labeled_cameras\':True,\'model\':\'TRELLIS image large\',\'gpu\':torch.cuda.get_device_name(0),\'steps\':16,\'texture_size\':2048,\'multi_view_conditioning\':f\'multi-image-{MULTI_VIEW_MODE}\'}\n\ndef generate(job_id,images,labels=None):\n    job=jobs[job_id]; start=time.monotonic(); folder=ROOT/job_id\n    try:\n        folder.mkdir()\n        digests=[]\n        for i,image in enumerate(images):\n            path=folder/f\'input-{i+1}.png\';image.save(path);digests.append(hashlib.sha256(path.read_bytes()).hexdigest())\n        params=dict(num_samples=1,seed=42,formats=[\'gaussian\',\'mesh\'],sparse_structure_sampler_params={\'steps\':16,\'cfg_strength\':7.5},slat_sampler_params={\'steps\':16,\'cfg_strength\':3.0})\n        if len(images)==1:\n            job[\'stage\']=\'Generating geometry and appearance from one reference image\'\n            outputs=pipeline.run(images[0],**params)\n            conditioning=\'single-image\'\n        else:\n            job[\'stage\']=f\'Blending {len(images)} views for geometry and appearance\'\n            samplers=(pipeline.sparse_structure_sampler,pipeline.slat_sampler)\n            original_methods=[sampler._inference_model for sampler in samplers]\n            try:\n                outputs=pipeline.run_multi_image(images,mode=MULTI_VIEW_MODE,**params)\n            except Exception:\n                # TRELLIS can leave its injected sampler installed when sampling raises.\n                for sampler,original in zip(samplers,original_methods):\n                    if \'_old_inference_model\' in vars(sampler):\n                        sampler._inference_model=original\n                        del sampler._old_inference_model\n                raise\n            conditioning=f\'multi-image-{MULTI_VIEW_MODE}\'\n        job[\'stage\']=\'Baking 2048 texture\'\n        source_mesh = outputs[\'mesh\'][0]\n        source_faces = len(source_mesh.faces)\n        mesh=postprocessing_utils.to_glb(outputs[\'gaussian\'][0],source_mesh,simplify=SIMPLIFY_RATIO,texture_size=2048)\n        job[\'stage\']=\'Matching reference cameras and applying visible photo detail\'\n        finish = refine(mesh, images, labels)\n        mesh.export(folder/\'model.glb\',tree_postprocessor=unlit)\n        details={\'model\':\'microsoft/TRELLIS-image-large\',\'seed\':42,\'steps\':16,\'texture_size\':2048,\'input_count\':len(images),\'conditioning\':conditioning,\'input_sha256\':digests,\'faces\':len(mesh.faces),\'vertices\':len(mesh.vertices),\'watertight\':bool(mesh.is_watertight),\'seconds\':round(time.monotonic()-start,1)}\n        details.update(source_faces=source_faces, simplify_ratio=SIMPLIFY_RATIO, **finish)\n        (folder/\'manifest.json\').write_text(json.dumps(details,indent=2))\n        job.update(status=\'complete\',stage=\'Model ready\',details=details)\n    except Exception as exc:\n        job.update(status=\'failed\',stage=\'Generation failed\',error=str(exc).replace(TOKEN,\'[redacted]\'))\n    finally:\n        job[\'elapsed\']=round(time.monotonic()-start,1)\n        try:\n            torch.cuda.empty_cache()\n        finally:\n            lock.release()\n\n@app.post(\'/jobs\')\nasync def create(request:Request):\n    if not lock.acquire(blocking=False):raise HTTPException(409,\'GPU is busy. Try again after the current model finishes.\')\n    try:\n        raw=bytearray()\n        async for chunk in request.stream():\n            if len(raw)+len(chunk)>32*1024*1024+4096:raise ValueError(\'Images exceed 24 MB total upload limit\')\n            raw.extend(chunk)\n        body=json.loads(raw); encoded=body[\'images\']\n        if not isinstance(encoded,list) or not 1<=len(encoded)<=6:raise ValueError(\'Choose one to six views of one object\')\n        labels=body.get(\'angles\',[\'unknown\']*len(encoded))\n        if not isinstance(labels,list) or len(labels)!=len(encoded) or any(label not in (\'front\',\'rear\',\'left\',\'right\',\'top\',\'bottom\',\'front_three_quarter\',\'rear_three_quarter\',\'front_three_quarter_nose_left\',\'front_three_quarter_nose_right\',\'rear_three_quarter_nose_left\',\'rear_three_quarter_nose_right\',\'oblique\',\'unknown\') for label in labels):\n            raise ValueError(\'Expected one angle per image\')\n        images=[];total=0\n        for index,item in enumerate(encoded,1):\n            data=base64.b64decode(item,validate=True);total+=len(data)\n            if total>24*1024*1024:raise ValueError(\'Images exceed 24 MB total upload limit\')\n            with Image.open(io.BytesIO(data),formats=[\'PNG\',\'JPEG\',\'WEBP\']) as image:\n                if image.width*image.height>16_000_000:raise ValueError(f\'View {index} exceeds 16 megapixels\')\n                if getattr(image,\'n_frames\',1)!=1:raise ValueError(f\'View {index} must be a still image\')\n                images.append(ImageOps.exif_transpose(image).convert(\'RGBA\'))\n        job_id=uuid.uuid4().hex\n        jobs[job_id]={\'id\':job_id,\'status\':\'running\',\'stage\':\'Preparing images\',\'elapsed\':0}\n        threading.Thread(target=generate,args=(job_id,images,labels),daemon=True).start()\n        return {\'id\':job_id}\n    except Exception as exc:\n        lock.release();raise HTTPException(400,str(exc))\n\n@app.get(\'/jobs/{job_id}\')\ndef status(job_id:str):\n    if job_id not in jobs:raise HTTPException(404)\n    return jobs[job_id]\n\n@app.get(\'/jobs/{job_id}/model\')\ndef model(job_id:str):\n    if job_id not in jobs or jobs[job_id][\'status\']!=\'complete\':raise HTTPException(404)\n    return FileResponse(ROOT/job_id/\'model.glb\',media_type=\'model/gltf-binary\')\n\nif __name__==\'__main__\':\n    import uvicorn\n    uvicorn.run(app,host=\'127.0.0.1\',port=7860,access_log=False)\n'
photo_source = '"""Register foreground photos to a generated textured mesh and refine its UV atlas."""\nimport math\n\nimport cv2\nimport numpy as np\nfrom numba import njit\nfrom PIL import Image\nfrom scipy.ndimage import distance_transform_edt, label as cc_label\nfrom scipy.spatial import cKDTree\n\nCARDINAL_VIEWS = {\'front\', \'rear\', \'left\', \'right\', \'top\', \'bottom\'}\n\n\n@njit\ndef raster(screen, faces, height, width):\n    """Nearest triangle and barycentric coordinates at each covered pixel."""\n    owner = np.full((height, width), -1, np.int32)\n    bary = np.zeros((height, width, 2), np.float32)\n    depth = np.full((height, width), -np.inf, np.float32)\n    for fi, (i, j, k) in enumerate(faces):\n        ax, ay, az = screen[i]\n        bx, by, bz = screen[j] - screen[i]\n        cx, cy, cz = screen[k] - screen[i]\n        det = bx*cy - by*cx\n        if abs(det) < 1e-9:\n            continue\n        for y in range(max(0, math.ceil(ay+min(0., by, cy))), min(height, math.floor(ay+max(0., by, cy))+1)):\n            for x in range(max(0, math.ceil(ax+min(0., bx, cx))), min(width, math.floor(ax+max(0., bx, cx))+1)):\n                u = ((x-ax)*cy-(y-ay)*cx)/det\n                v = (bx*(y-ay)-by*(x-ax))/det\n                z = az+u*bz+v*cz\n                if u >= 0 and v >= 0 and u+v <= 1 and z > depth[y,x]:\n                    owner[y,x] = fi\n                    bary[y,x,0], bary[y,x,1] = u,v\n                    depth[y,x] = z\n    return owner, bary, depth\n\n\n@njit\ndef raster_uv(uv, faces, height, width):\n    """Conservative UV triangle rasterizer with a 1-texel border skirt for thin islands."""\n    owner = np.full((height, width), -1, np.int32)\n    bary = np.zeros((height, width, 2), np.float32)\n    exact = np.zeros((height, width), np.uint8)\n    best_dist = np.full((height, width), np.inf, np.float32)\n    for fi, (i, j, k) in enumerate(faces):\n        ax, ay = uv[i, 0], uv[i, 1]\n        bx, by = uv[j, 0] - ax, uv[j, 1] - ay\n        cx, cy = uv[k, 0] - ax, uv[k, 1] - ay\n        det = bx * cy - by * cx\n        if abs(det) < 1e-9:\n            continue\n        y0 = max(0, math.floor(ay + min(0.0, by, cy)) - 1)\n        y1 = min(height, math.ceil(ay + max(0.0, by, cy)) + 2)\n        x0 = max(0, math.floor(ax + min(0.0, bx, cx)) - 1)\n        x1 = min(width, math.ceil(ax + max(0.0, bx, cx)) + 2)\n        for y in range(y0, y1):\n            for x in range(x0, x1):\n                u = ((x - ax) * cy - (y - ay) * cx) / det\n                v = (bx * (y - ay) - by * (x - ax)) / det\n                if u >= 0.0 and v >= 0.0 and u + v <= 1.0:\n                    owner[y, x] = fi\n                    bary[y, x, 0], bary[y, x, 1] = u, v\n                    exact[y, x] = 1\n                    best_dist[y, x] = 0.0\n                elif exact[y, x] == 0 and u >= -0.35 and v >= -0.35 and u + v <= 1.35:\n                    uc = max(0.0, u)\n                    vc = max(0.0, v)\n                    uv_sum = uc + vc\n                    if uv_sum > 1.0:\n                        uc /= uv_sum\n                        vc /= uv_sum\n                    dist = (u - uc) ** 2 + (v - vc) ** 2\n                    if dist < best_dist[y, x]:\n                        owner[y, x] = fi\n                        bary[y, x, 0], bary[y, x, 1] = uc, vc\n                        best_dist[y, x] = dist\n    return owner, bary\n\n\ndef camera(vertices, yaw, elevation):\n    a,b = np.deg2rad([yaw,elevation])\n    right = np.array([np.cos(a),0,-np.sin(a)])\n    direction = np.array([np.sin(a)*np.cos(b),np.sin(b),np.cos(a)*np.cos(b)])\n    basis = np.stack([right,np.cross(right,direction),direction]).astype(\'f\')\n    projected = vertices @ basis.T\n    extent = np.maximum(np.ptp(projected[:,:2],axis=0),1e-6)\n    projected[:,:2] = (projected[:,:2]-projected[:,:2].min(0))/extent\n    return projected, direction, extent[0]/extent[1]\n\n\ndef interpolate(values, triangles, weights):\n    a,b,c = values[triangles].transpose(1,0,2)\n    return a+(b-a)*weights[:,0:1]+(c-a)*weights[:,1:2]\n\n\ndef foreground(photo, session):\n    """Extract cleaned RGBA object crop with border decontamination and edge-clip flags."""\n    photo = photo.copy()\n    photo.thumbnail((1280,1280),Image.Resampling.LANCZOS)\n    if photo.mode != \'RGBA\' or photo.getchannel(\'A\').getextrema()[0] == 255:\n        from rembg import remove\n        photo = remove(photo, session=session)\n    rgba = np.asarray(photo.convert(\'RGBA\')).copy()\n    mask = rgba[:, :, 3] > 128\n    comp, n_comp = cc_label(mask)\n    if n_comp > 1:\n        areas = np.bincount(comp.ravel())\n        areas[0] = 0\n        largest = int(areas.max())\n        keep = areas[comp] >= 0.02 * largest\n        rgba[~keep, 3] = 0\n        mask = rgba[:, :, 3] > 128\n    ys, xs = np.where(mask)\n    if ys.size == 0:\n        return None, (False,)*4\n    y0, y1 = int(ys.min()), int(ys.max())\n    x0, x1 = int(xs.min()), int(xs.max())\n    if min(x1 - x0 + 1, y1 - y0 + 1) < 8:\n        return None, (False,)*4\n    crop = rgba[y0:y1 + 1, x0:x1 + 1].copy()\n    opaque = crop[:, :, 3] >= 235\n    if opaque.any() and not opaque.all():\n        near = distance_transform_edt(~opaque, return_distances=False, return_indices=True)\n        crop[~opaque, :3] = crop[near[0][~opaque], near[1][~opaque], :3]\n    m = crop[:, :, 3] > 128\n    sides = (m[:, 0], m[:, -1], m[0], m[-1])\n    at_edge = (x0 == 0, x1 + 1 == rgba.shape[1], y0 == 0, y1 + 1 == rgba.shape[0])\n    return crop, tuple(bool(edge and side.mean() > .08) for edge, side in zip(at_edge, sides))\n\n\ndef visible_windows(edges, photo_aspect, model_aspect):\n    """Candidate regions of the projected mesh visible inside a clipped photo."""\n    left,right,top,bottom = edges\n    horizontal,vertical = left or right,top or bottom\n    if horizontal == vertical:\n        yield (0.,1.,0.,1.)\n        return\n    span = float(min(1., photo_aspect/model_aspect if horizontal else model_aspect/photo_aspect))\n    low,high = (left,right) if horizontal else (top,bottom)\n    starts = np.linspace(0,1-span,13) if low and high else [1-span if low else 0.]\n    for start in starts:\n        yield (float(start),span,0.,1.) if horizontal else (0.,1.,float(start),span)\n\n\ndef joint_cameras(scores, labels):\n    """Choose the globally consistent camera pose assignment across labeled reference views."""\n    offsets = {\'front\': 0, \'right\': -90, \'rear\': 180, \'left\': 90}\n\n    def allowed(label, front, candidate):\n        yaw,elev = candidate[1:3]\n        base,_,nose = label.partition(\'_nose_\')\n        signed = (yaw-front-offsets.get(base,0)+180)%360-180\n        delta = abs(signed)\n        if nose==\'left\' and signed<=0 or nose==\'right\' and signed>=0:\n            return False\n        if base in offsets:\n            return delta == 0 and elev == 0\n        if base in (\'top\',\'bottom\'):\n            expected_elev = 90 if base == \'top\' else -90\n            return elev == expected_elev and ((yaw - front) % 90 == 0)\n        if base.endswith(\'three_quarter\'):\n            return (20 <= delta <= 70 if base.startswith(\'front\') else 110 <= delta <= 160) and abs(elev) <= 50\n        return True\n\n    # The generated object\'s forward axis is unknown. Score complete assignments\n    # around one shared heading, including the side profiles that reveal its nose.\n    best = None\n    for front in range(0,360,30):\n        picks = [max((c for c in row if allowed(label,front,c)), default=None)\n                 for row,label in zip(scores,labels)]\n        quality = sum(p[0] if p else -1 for p in picks)\n        if best is None or quality > best[0]:\n            best = quality,picks\n    return best[1]\n\n\ndef _wheel_centroids(mask):\n    """Locate left and right ground-contact wheel horizontal centroids from a side silhouette mask."""\n    ph, pw = mask.shape\n    bot = np.array(\n        [float(np.flatnonzero(mask[:, x])[-1]) if mask[:, x].any() else 0.0 for x in range(pw)],\n        dtype=np.float32,\n    )\n    centers = []\n    for a, b in ((int(0.05 * pw), int(0.45 * pw)), (int(0.55 * pw), int(0.95 * pw))):\n        if b <= a:\n            return None\n        sub = bot[a:b]\n        w = np.maximum(0.0, sub - np.quantile(sub, 0.78)) ** 2\n        w_sum = float(w.sum())\n        if w_sum <= 0.0:\n            return None\n        xs = np.arange(a, b, dtype=np.float32)\n        centers.append(float((xs * w).sum() / w_sum))\n    return centers[0], centers[1]\n\n\ndef _symmetry_axis(m_mask, pic, axis):\n    """Return (m_sym, p_sym, body_scale) when mesh silhouette and photo RGB are bilaterally symmetric."""\n    m_s = m_mask if axis == 1 else m_mask.T\n    gray = cv2.cvtColor(pic[:, :, :3], cv2.COLOR_RGB2GRAY).astype(np.float32)\n    p_hi = pic[:, :, 3] > 180\n    p_al = pic[:, :, 3] > 128\n    g_s = gray if axis == 1 else gray.T\n    pm_s = p_hi if axis == 1 else p_hi.T\n    pa_s = p_al if axis == 1 else p_al.T\n\n    orth_len, lat_len = m_s.shape\n    m_centers, m_in, p_in = [], [], []\n    o0, o1 = int(0.10 * orth_len), int(0.90 * orth_len)\n    i0, i1 = int(0.15 * orth_len), int(0.85 * orth_len)\n    for o in range(o0, o1):\n        idx = np.flatnonzero(m_s[o])\n        if idx.size > 0:\n            m_centers.append(0.5 * float(idx[0] + idx[-1]))\n            if i0 <= o < i1:\n                m_in.append(float(idx[-1] - idx[0]))\n        if i0 <= o < i1:\n            pidx = np.flatnonzero(pa_s[o])\n            if pidx.size > 0:\n                p_in.append(float(pidx[-1] - pidx[0]))\n    if len(m_centers) < 10 or not m_in or not p_in:\n        return None\n\n    m_sym = float(np.median(m_centers))\n    ks = np.arange(lat_len)\n    ms_mesh = np.rint(2.0 * m_sym - ks).astype(int)\n    ok_m = (ms_mesh >= 0) & (ms_mesh < lat_len)\n    mirror_iou = float((m_s[:, ks[ok_m]] & m_s[:, ms_mesh[ok_m]]).sum()) / max(\n        int((m_s[:, ks[ok_m]] | m_s[:, ms_mesh[ok_m]]).sum()), 1\n    )\n    if mirror_iou < 0.86:\n        return None\n\n    min_ov = 0.35 * float(pm_s.sum())\n    best_sym = None\n    for c2 in range(int(round(0.70 * lat_len)), int(round(1.30 * lat_len)) + 1):\n        c = 0.5 * c2\n        ms = np.rint(2.0 * c - ks).astype(int)\n        ok = (ms >= 0) & (ms < lat_len) & (ks < ms)\n        k1, k2 = ks[ok], ms[ok]\n        ov = pm_s[:, k1] & pm_s[:, k2]\n        if float(ov.sum()) < min_ov:\n            continue\n        v1, v2 = g_s[:, k1][ov], g_s[:, k2][ov]\n        err = float(np.abs(v1 - v2).mean())\n        if best_sym is None or err < best_sym[0]:\n            best_sym = (err, c, v1, v2)\n    if best_sym is None:\n        return None\n    _, p_sym, v1, v2 = best_sym\n    if v1.size < 16 or float(v1.std()) < 1.0 or float(v2.std()) < 1.0:\n        return None\n    corr = float(np.corrcoef(v1, v2)[0, 1])\n    if not np.isfinite(corr) or corr < 0.72:\n        return None\n\n    m_arr = np.asarray(m_in, dtype=np.float32)\n    p_arr = np.asarray(p_in, dtype=np.float32)\n    m_keep = m_arr[m_arr <= np.quantile(m_arr, 0.85)]\n    p_keep = p_arr[p_arr <= np.quantile(p_arr, 0.85)]\n    m_span = float(np.median(m_keep)) if m_keep.size else float(np.median(m_arr))\n    p_span = float(np.median(p_keep)) if p_keep.size else float(np.median(p_arr))\n    body_scale = float(np.clip(p_span / max(m_span, 1.0), 0.92, 1.18))\n    return m_sym, p_sym, body_scale\n\n\ndef _align_symmetric(projected, m_small, p_small, raw_iou, shape_hw, sym, lat_axis, min_gain):\n    """Lock the lateral RGB symmetry centerline while optimizing orthogonal alignment."""\n    ph, pw = shape_hw\n    sh_grid, sw_grid = p_small.shape\n    cx_s, cy_s = 0.5 * (sw_grid - 1), 0.5 * (sh_grid - 1)\n    m_sym, p_sym, body_scale = sym\n\n    if lat_axis == 1:\n        m_g = m_sym * (sw_grid - 1) / max(pw - 1, 1)\n        p_g = p_sym * (sw_grid - 1) / max(pw - 1, 1)\n        sx_vals = np.linspace(0.95, 1.17, 12)\n        sy_lo, sy_hi = 0.94, 1.18\n        sy_vals = np.linspace(0.95, 1.17, 12)\n        t_vals = np.linspace(-0.08, 0.08, 9)\n        min_center_shift = 1.0\n    else:\n        m_g = m_sym * (sh_grid - 1) / max(ph - 1, 1)\n        p_g = p_sym * (sh_grid - 1) / max(ph - 1, 1)\n        sx_vals = np.linspace(0.95, 1.15, 11)\n        sy_lo = max(0.94, body_scale - 0.03)\n        sy_hi = min(1.16, body_scale + 0.03)\n        sy_vals = np.linspace(sy_lo, sy_hi, 5)\n        t_vals = np.linspace(-0.06, 0.06, 9)\n        min_center_shift = 1.5\n\n    def score_pose(sx, sy, t_orth):\n        if lat_axis == 1:\n            mat = np.array(\n                [[sx, 0.0, p_g - sx * m_g], [0.0, sy, cy_s * (1.0 - sy) + t_orth * sh_grid]],\n                dtype=np.float32,\n            )\n        else:\n            mat = np.array(\n                [[sx, 0.0, cx_s * (1.0 - sx) + t_orth * sw_grid], [0.0, sy, p_g - sy * m_g]],\n                dtype=np.float32,\n            )\n        warped = cv2.warpAffine(m_small, mat, (sw_grid, sh_grid), flags=cv2.INTER_NEAREST) > 0\n        return float((warped & p_small).sum()) / max(int((warped | p_small).sum()), 1)\n\n    best = (-1.0, 1.0, 1.0, 0.0)\n    for sx in sx_vals:\n        for sy in sy_vals:\n            for t_orth in t_vals:\n                iou = score_pose(float(sx), float(sy), float(t_orth))\n                if iou > best[0]:\n                    best = (iou, float(sx), float(sy), float(t_orth))\n\n    for step in (0.01, 0.005):\n        _, bsx, bsy, bt = best\n        for dsx in (-step, 0.0, step):\n            sx = float(np.clip(bsx + dsx, 0.94, 1.18))\n            for dsy in (-step, 0.0, step):\n                sy = float(np.clip(bsy + dsy, sy_lo, sy_hi))\n                for dt in (-step, 0.0, step):\n                    t_orth = float(np.clip(bt + dt, -0.09, 0.09))\n                    iou = score_pose(sx, sy, t_orth)\n                    if iou > best[0]:\n                        best = (iou, sx, sy, t_orth)\n\n    best_iou, bsx, bsy, bt = best\n    if best_iou < raw_iou + min_gain and abs(p_sym - m_sym) < min_center_shift:\n        return projected, False\n\n    aligned = projected.copy()\n    cx, cy = 0.5 * (pw - 1), 0.5 * (ph - 1)\n    if lat_axis == 1:\n        aligned[:, 0] = (aligned[:, 0] - m_sym) * bsx + p_sym\n        aligned[:, 1] = (aligned[:, 1] - cy) * bsy + cy + bt * ph\n    else:\n        aligned[:, 0] = (aligned[:, 0] - cx) * bsx + cx + bt * pw\n        aligned[:, 1] = (aligned[:, 1] - m_sym) * bsy + p_sym\n    return aligned, True\n\n\ndef _align_silhouette(projected, m_mask, pic, base_label, min_gain):\n    """Align unclipped orthographic projections with symmetry locking or 4-DOF silhouette IoU."""\n    ph, pw = m_mask.shape\n    p_mask = pic[:, :, 3] > 128\n    sw_grid = min(pw, 128)\n    sh_grid = max(16, int(round(ph * sw_grid / max(pw, 1))))\n    m_small = (cv2.resize(m_mask.astype(\'uint8\'), (sw_grid, sh_grid), interpolation=cv2.INTER_NEAREST) > 0).astype(\'uint8\')\n    p_small = cv2.resize(p_mask.astype(\'uint8\'), (sw_grid, sh_grid), interpolation=cv2.INTER_NEAREST) > 0\n    raw_iou = float(((m_small > 0) & p_small).sum()) / max(int(((m_small > 0) | p_small).sum()), 1)\n\n    lat_axis = 1 if base_label in (\'front\', \'rear\') else (0 if pw >= ph else 1)\n    sym = _symmetry_axis(m_mask, pic, lat_axis)\n    if sym is not None:\n        return _align_symmetric(projected, m_small, p_small, raw_iou, (ph, pw), sym, lat_axis, min_gain)\n\n    cx_s, cy_s = 0.5 * (sw_grid - 1), 0.5 * (sh_grid - 1)\n\n    def score_pose(sx, sy, tx_frac, ty_frac):\n        mat = np.array(\n            [\n                [sx, 0.0, cx_s * (1.0 - sx) + tx_frac * sw_grid],\n                [0.0, sy, cy_s * (1.0 - sy) + ty_frac * sh_grid],\n            ],\n            dtype=np.float32,\n        )\n        warped = cv2.warpAffine(m_small, mat, (sw_grid, sh_grid), flags=cv2.INTER_NEAREST) > 0\n        return float((warped & p_small).sum()) / max(int((warped | p_small).sum()), 1)\n\n    best = (-1.0, 1.0, 1.0, 0.0, 0.0)\n    for sx in np.linspace(0.95, 1.17, 12):\n        for sy in np.linspace(0.95, 1.17, 12):\n            for ty_frac in np.linspace(-0.08, 0.08, 9):\n                iou = score_pose(float(sx), float(sy), 0.0, float(ty_frac))\n                if iou > best[0]:\n                    best = (iou, float(sx), float(sy), 0.0, float(ty_frac))\n\n    prev_tx = best[3]\n    _, bsx, bsy, _, bty = best\n    for tx_frac in np.linspace(-0.08, 0.08, 9):\n        iou = score_pose(bsx, bsy, float(tx_frac), bty)\n        if iou > best[0]:\n            best = (iou, bsx, bsy, float(tx_frac), bty)\n    if best[3] != prev_tx:\n        _, bsx, bsy, btx, _ = best\n        for ty_frac in np.linspace(-0.08, 0.08, 9):\n            iou = score_pose(bsx, bsy, btx, float(ty_frac))\n            if iou > best[0]:\n                best = (iou, bsx, bsy, btx, float(ty_frac))\n\n    for step in (0.01, 0.005):\n        _, bsx, bsy, btx, bty = best\n        for dsx in (-step, 0.0, step):\n            sx = float(np.clip(bsx + dsx, 0.94, 1.18))\n            for dsy in (-step, 0.0, step):\n                sy = float(np.clip(bsy + dsy, 0.94, 1.18))\n                for dtx in (-step, 0.0, step):\n                    tx = float(np.clip(btx + dtx, -0.09, 0.09))\n                    for dty in (-step, 0.0, step):\n                        ty = float(np.clip(bty + dty, -0.09, 0.09))\n                        iou = score_pose(sx, sy, tx, ty)\n                        if iou > best[0]:\n                            best = (iou, sx, sy, tx, ty)\n\n    best_iou, bsx, bsy, btx, bty = best\n    if best_iou < raw_iou + min_gain:\n        return projected, False\n    aligned = projected.copy()\n    cx, cy = 0.5 * (pw - 1), 0.5 * (ph - 1)\n    aligned[:, 0] = (aligned[:, 0] - cx) * bsx + cx + btx * pw\n    aligned[:, 1] = (aligned[:, 1] - cy) * bsy + cy + bty * ph\n    return aligned, True\n\n\ndef _align_side_wheelbase(projected, m_mask, p_mask):\n    """Align unclipped side projections via rigid horizontal wheelbase affine fit."""\n    ph, pw = m_mask.shape\n    mc = _wheel_centroids(m_mask)\n    pc = _wheel_centroids(p_mask)\n    if mc is None or pc is None or (mc[1] - mc[0]) <= 1.0:\n        return projected, False\n    sx = (pc[1] - pc[0]) / (mc[1] - mc[0])\n    tx = pc[0] - sx * mc[0]\n    if not (0.92 <= sx <= 1.12 and abs(tx) <= 0.08 * pw and (abs(sx - 1.0) > 0.02 or abs(tx) > 2.5)):\n        return projected, False\n    raw_iou = float((m_mask & p_mask).sum()) / max(int((m_mask | p_mask).sum()), 1)\n    mat = np.array([[sx, 0.0, tx], [0.0, 1.0, 0.0]], dtype=np.float32)\n    warped = cv2.warpAffine(m_mask.astype(\'uint8\'), mat, (pw, ph), flags=cv2.INTER_NEAREST) > 0\n    new_iou = float((warped & p_mask).sum()) / max(int((warped | p_mask).sum()), 1)\n    if new_iou < raw_iou - 0.015:\n        return projected, False\n    aligned = projected.copy()\n    aligned[:, 0] = aligned[:, 0] * sx + tx\n    return aligned, True\n\n\ndef _align_projection(projected, depth, pic, base_label, window):\n    """Align orthographic projections to compensate protruding appendages or cropped mirrors."""\n    if window[1] < 0.90 or window[3] < 0.90 or base_label not in CARDINAL_VIEWS:\n        return projected, False\n    m_mask = np.isfinite(depth)\n    if base_label in (\'front\', \'rear\'):\n        return _align_silhouette(projected, m_mask, pic, base_label, 0.020)\n    if base_label in (\'top\', \'bottom\'):\n        return _align_silhouette(projected, m_mask, pic, base_label, 0.015)\n    return _align_side_wheelbase(projected, m_mask, pic[:, :, 3] > 128)\n\n\ndef _depth_reach(base_label, q_z, z_min, z_span, u_img, v_img, pw, ph):\n    """Down-weight far-side grazing regions where orthogonal views own the object\'s flanks or end fascias."""\n    z_norm = np.clip((q_z - z_min) / z_span, 0.0, 1.0).astype(np.float32)\n    u_norm = np.abs(2.0 * (u_img / max(float(pw - 1), 1.0)) - 1.0).astype(np.float32)\n    if base_label in (\'front\', \'rear\'):\n        flank = np.clip((u_norm - 0.52) / 0.28, 0.0, 1.0)\n        z_cut = 0.46 + 0.26 * flank\n        return np.clip((z_norm - (z_cut - 0.14)) / 0.14, 0.005, 1.0).astype(np.float32)\n    if base_label in (\'left\', \'right\'):\n        return np.clip((z_norm - 0.34) / 0.16, 0.005, 1.0).astype(np.float32)\n    if base_label in (\'top\', \'bottom\'):\n        base_r = np.clip((z_norm - 0.28) / 0.18, 0.01, 1.0).astype(np.float32)\n        long_norm = (\n            u_norm\n            if pw >= ph\n            else np.abs(2.0 * (v_img / max(float(ph - 1), 1.0)) - 1.0).astype(np.float32)\n        )\n        end_cap = np.clip((long_norm - 0.56) / 0.20, 0.0, 1.0)\n        low_fascia = np.clip((0.76 - z_norm) / 0.22, 0.0, 1.0)\n        taper = 1.0 - 0.96 * end_cap * low_fascia\n        return (base_r * taper).astype(np.float32)\n    return np.ones_like(z_norm, dtype=np.float32)\n\n\ndef _sample_rgba_bilinear(pic_f, u_img, v_img):\n    """Sample RGBA float32 image bilinearly at subpixel image coordinates."""\n    ph, pw = pic_f.shape[:2]\n    uc = np.clip(u_img, 0.0, float(pw - 1))\n    vc = np.clip(v_img, 0.0, float(ph - 1))\n    ix0 = np.floor(uc).astype(int)\n    iy0 = np.floor(vc).astype(int)\n    ix1 = np.minimum(ix0 + 1, pw - 1)\n    iy1 = np.minimum(iy0 + 1, ph - 1)\n    du = (uc - ix0)[:, None].astype(np.float32)\n    dv = (vc - iy0)[:, None].astype(np.float32)\n    w00 = (1.0 - du) * (1.0 - dv)\n    w01 = du * (1.0 - dv)\n    w10 = (1.0 - du) * dv\n    w11 = du * dv\n    return (\n        pic_f[iy0, ix0] * w00\n        + pic_f[iy0, ix1] * w01\n        + pic_f[iy1, ix0] * w10\n        + pic_f[iy1, ix1] * w11\n    )\n\n\ndef _inpaint_unobserved(fused, mix, vertices, faces, owner, bary, valid, model_diag):\n    """Fill shadowed or grazing texels from nearby observed 3D surface colors and dilate UV seams."""\n    ys, xs = np.where(valid)\n    good_idx = np.flatnonzero(mix[ys, xs] >= 0.65)\n    bad_idx = np.flatnonzero(mix[ys, xs] < 0.65)\n    if good_idx.size >= 16 and bad_idx.size > 0:\n        pos = interpolate(vertices, faces[owner[ys, xs]], bary[ys, xs])\n        stride = max(1, good_idx.size // 60000)\n        sup = good_idx[::stride]\n        tree = cKDTree(pos[sup])\n        sup_rgb = fused[ys[sup], xs[sup]]\n        radius = max(model_diag * 0.16, 1e-4)\n        k = min(12, sup.size)\n        chunks = max(1, math.ceil(bad_idx.size / 65536))\n        for chunk in np.array_split(bad_idx, chunks):\n            ds, js = tree.query(pos[chunk], k=k)\n            if k == 1:\n                ds = ds[:, None]\n                js = js[:, None]\n            fill_rgb = np.median(sup_rgb[js], axis=1).astype(np.float32)\n            fill_blend = np.exp(-((ds[:, 0] / radius) ** 2)).astype(np.float32)\n            existing = mix[ys[chunk], xs[chunk]]\n            improve = fill_blend > existing\n            if not improve.any():\n                continue\n            sel = chunk[improve]\n            cy, cx = ys[sel], xs[sel]\n            bw = (fill_blend[improve] * (1.0 - existing[improve]))[:, None]\n            fused[cy, cx] = fused[cy, cx] * (1.0 - bw) + fill_rgb[improve] * bw\n            mix[cy, cx] = fill_blend[improve]\n    if valid.any() and not valid.all():\n        near = distance_transform_edt(~valid, return_distances=False, return_indices=True)\n        fused[~valid] = fused[near[0][~valid], near[1][~valid]]\n\n\ndef refine(mesh, images, labels=None):\n    """Register reference photos to mesh cameras and project high-resolution photo detail into UV atlas."""\n    mat = mesh.visual.material\n    atlas = np.asarray(mat.baseColorTexture.convert(\'RGB\'))\n    h,w = atlas.shape[:2]\n    uv = np.asarray(mesh.visual.uv,dtype=\'f\').copy()\n    uv[:,1] = 1-uv[:,1]\n    uv *= [w-1,h-1]\n    vertices = np.asarray(mesh.vertices,dtype=\'f\')\n    faces = np.asarray(mesh.faces,dtype=np.int32)\n    session = None\n    if any(im.mode != \'RGBA\' or im.getchannel(\'A\').getextrema()[0] == 255 for im in images):\n        from rembg import new_session\n        session = new_session(\'u2net\', providers=[\'CPUExecutionProvider\'])\n    photos,edges = zip(*(foreground(im, session) for im in images))\n    del session\n    refs = [None if p is None else cv2.resize(p,(64,64),interpolation=cv2.INTER_AREA) for p in photos]\n    scores = [[] for _ in photos]\n    grid_y,grid_x = np.indices((64,64),dtype=\'f\')\n\n    def evaluate(yaw,elev,indices):\n        screen,_,aspect = camera(vertices,yaw,elev)\n        screen[:,:2] *= 63\n        face,bary,_ = raster(screen,faces,64,64)\n        hit = face>=0\n        tex = interpolate(uv,faces[face[hit]],bary[hit]).round().astype(int)\n        render = np.zeros((64,64,3),np.float32)\n        render[hit] = atlas[tex[:,1].clip(0,h-1),tex[:,0].clip(0,w-1)]\n        for i in indices:\n            ref = refs[i]\n            if ref is None:\n                continue\n            mask = ref[:,:,3]>128\n            ratio = photos[i].shape[1]/photos[i].shape[0]\n            best = None\n            for window in visible_windows(edges[i],ratio,aspect):\n                sx,sw,sy,sh = window\n                if sw==sh==1:\n                    visible,colors = hit,render\n                else:\n                    mx,my = grid_x*sw+63*sx,grid_y*sh+63*sy\n                    visible = cv2.remap(hit.astype(\'uint8\'),mx,my,cv2.INTER_NEAREST)>0\n                    colors = cv2.remap(render,mx,my,cv2.INTER_LINEAR)\n                overlap = mask & visible\n                if not overlap.any():\n                    continue\n                iou = overlap.sum()/(mask|visible).sum()\n                target = ref[:,:,:3][overlap].astype(\'f\')\n                projected = colors[overlap]\n                gain = np.clip((target.mean()+1)/(projected.mean()+1),.6,1.7)\n                error = np.abs(target-projected*gain).mean()/255\n                aspect_error = abs(np.log(aspect*sw/(ratio*sh)))\n                score = iou-.72*error-.18*min(aspect_error,2)\n                candidate = (score,yaw,elev,iou,error,window)\n                if best is None or score>best[0]:\n                    best = candidate\n            if best is not None:\n                scores[i].append(best)\n\n    for elev in (-90,-45,-20,0,20,45,90):\n        for yaw in range(0,360,30):\n            evaluate(yaw,elev,range(len(refs)))\n    supplied_labels = labels or [\'unknown\']*len(images)\n    labels = list(supplied_labels)\n    for i,(ref,label) in enumerate(zip(refs,labels)):\n        if ref is None or \'three_quarter\' not in label:\n            continue\n        # Cross-check oblique predictions on bilaterally symmetric end views.\n        gray = cv2.cvtColor(ref[:,:,:3],cv2.COLOR_RGB2GRAY).astype(\'f\')\n        mask = ref[:,:,3]>128\n        overlap = mask & mask[:,::-1]\n        values, mirror = gray[overlap],gray[:,::-1][overlap]\n        if (overlap.sum() > .87*mask.sum() and values.size and values.std()>5\n                and np.corrcoef(values,mirror)[0,1]>.72):\n            labels[i] = label.split(\'_\')[0]\n    for i,match in enumerate(joint_cameras(scores,labels)):\n        if match and labels[i].partition(\'_nose_\')[0] not in CARDINAL_VIEWS:\n            a,b = match[1:3]\n            for da in (-12,0,12):\n                for db in (-8,0,8):\n                    evaluate(a+da,np.clip(b+db,-90,90),[i])\n\n    matches = joint_cameras(scores, labels)\n    report = []\n    selected = []\n    for pic, match, label, candidates in zip(photos, matches, labels, scores):\n        if not match:\n            report.append(dict(applied=False, label=label, reason=\'no eligible camera\'))\n            selected.append(None)\n            continue\n        score, yaw, elev, iou, error, window = match\n        if label in (\'unknown\', \'oblique\') and abs(elev) < 65 and any(\n            abs((other[1] - yaw + 180) % 360 - 180) > 140 and abs(other[2] - elev) < 25\n            and other[0] >= score - .025 for other in candidates\n        ):\n            report.append(dict(applied=False, label=label, reason=\'ambiguous opposite directions\'))\n            selected.append(None)\n            continue\n        applied = bool(iou >= .70 and score >= .52 and error < .36)\n        item = dict(\n            applied=applied,\n            label=label,\n            yaw=float(yaw),\n            elevation=float(elev),\n            silhouette_iou=round(float(iou), 3),\n            score=round(float(score), 3),\n        )\n        if applied:\n            item[\'visible_window\'] = [float(v) for v in window]\n            selected.append((pic, match, label))\n        else:\n            selected.append(None)\n        report.append(item)\n\n    cardinal_applied = sum(\n        1 for item in report\n        if item.get(\'applied\') and item[\'label\'].partition(\'_nose_\')[0] in CARDINAL_VIEWS\n    )\n\n    owner, bary = raster_uv(uv, faces, h, w)\n    v_normals = np.asarray(mesh.vertex_normals, dtype=\'f\')\n    f_normals = np.asarray(mesh.face_normals, dtype=\'f\')\n    model_diag = float(np.linalg.norm(np.ptp(vertices, axis=0)))\n    tolerance = model_diag * 0.012\n    weighted = np.zeros((h, w, 3), np.float32)\n    weights = np.zeros((h, w), np.float32)\n    max_conf = np.zeros((h, w), np.float32)\n\n    for entry in selected:\n        if entry is None:\n            continue\n        pic, match, label = entry\n        base_label = label.partition(\'_nose_\')[0]\n        is_cardinal = base_label in CARDINAL_VIEWS\n        view_priority = np.float32(0.002 if (not is_cardinal and cardinal_applied >= 4) else 1.0)\n        conf_scale = np.float32(1.0 if (is_cardinal or cardinal_applied < 4) else 0.75)\n\n        score, yaw, elev, _, _, window = match\n        ph, pw = pic.shape[:2]\n        pic_f = pic.astype(np.float32)\n        projected, direction, _ = camera(vertices, yaw, elev)\n        direction = np.asarray(direction, dtype=\'f\')\n        sx, sw, sy, sh = window\n        if base_label not in CARDINAL_VIEWS or sw < 0.90 or sh < 0.90:\n            projected[:, :2] = (projected[:, :2] - [sx, sy]) / [sw, sh]\n        projected[:, :2] *= [pw - 1, ph - 1]\n        _, _, depth = raster(projected, faces, ph, pw)\n        projected, changed = _align_projection(projected, depth, pic, base_label, window)\n        if changed:\n            _, _, depth = raster(projected, faces, ph, pw)\n\n        hit = np.isfinite(depth)\n        if hit.any():\n            pad = float(np.ptp(depth[hit])) + 10.0\n            hi_sent = float(depth[hit].max()) + pad\n            lo_sent = float(depth[hit].min()) - pad\n            d_min = cv2.erode(np.where(hit, depth, hi_sent).astype(\'f\'), np.ones((3, 3), np.uint8))\n            d_max = cv2.dilate(np.where(hit, depth, lo_sent).astype(\'f\'), np.ones((3, 3), np.uint8))\n            d_min_ok = d_min < hi_sent - 0.5 * pad\n            d_max_ok = d_max > lo_sent + 0.5 * pad\n            jump = np.where(hit & d_min_ok & d_max_ok, np.maximum(d_max - d_min, 0.0), 0.0)\n        else:\n            d_min = np.full_like(depth, 1e6)\n            d_min_ok = np.zeros_like(hit)\n            jump = np.zeros_like(depth)\n        jump_map = np.clip(\n            1.0 - np.maximum(jump - tolerance * 2.0, 0.0) / max(tolerance * 8.0, 1e-6),\n            0.15,\n            1.0,\n        ).astype(\'f\')\n\n        dist_in = distance_transform_edt(pic[:, :, 3] > 140).astype(\'f\')\n        border_pad = max(3.0, min(ph, pw) * 0.018)\n        border_map = (np.clip(dist_in / border_pad, 0.04, 1.0).astype(\'f\')) ** 2\n\n        z_min = float(projected[:, 2].min())\n        z_span = max(float(np.ptp(projected[:, 2])), 1e-6)\n        score_pow = np.float32(max(float(score), 0.0) ** 4)\n\n        for row in range(0, h, 64):\n            block = owner[row:row + 64]\n            yy, xx = np.where(block >= 0)\n            if yy.size == 0:\n                continue\n            yy += row\n            fi = owner[yy, xx]\n            tri = faces[fi]\n            frac = bary[yy, xx]\n            q = interpolate(projected, tri, frac)\n            u_img, v_img = q[:, 0], q[:, 1]\n            inside = (u_img >= 0.0) & (u_img <= pw - 1.0) & (v_img >= 0.0) & (v_img <= ph - 1.0)\n            px = np.clip(np.rint(u_img).astype(int), 0, pw - 1)\n            py = np.clip(np.rint(v_img).astype(int), 0, ph - 1)\n\n            d_lo = d_min[py, px]\n            d_hi = depth[py, px]\n            d_lo_fin = d_min_ok[py, px]\n            d_hi_fin = hit[py, px]\n            dz_front = np.where(d_lo_fin, d_lo - q[:, 2], np.inf)\n            dz_exact = np.where(d_hi_fin, np.where(d_hi_fin, d_hi, 0.0) - q[:, 2], np.inf)\n            front_shell = inside & (dz_front <= tolerance) & (dz_exact <= tolerance * 2.5)\n            cavity_mask = inside & d_hi_fin & (dz_exact > 0.0) & (dz_exact < tolerance * 14.0)\n            dz_cavity = np.clip(np.where(cavity_mask, dz_exact, 0.0), 0.0, tolerance * 14.0)\n            cavity_vis = np.where(cavity_mask, 0.015 * np.exp(-dz_cavity / max(tolerance * 4.0, 1e-6)), 0.0)\n            vis_factor = np.where(front_shell, 1.0, cavity_vis).astype(\'f\')\n\n            facing_v = interpolate(v_normals, tri, frac) @ direction\n            facing_f = f_normals[fi] @ direction\n            facing_pos = np.maximum(np.maximum(facing_v, 0.75 * facing_f), 0.0)\n            facing_eff = np.maximum(np.where(front_shell, np.maximum(facing_pos, 0.06), facing_pos), 0.0).astype(\'f\')\n            if base_label in (\'front\', \'rear\'):\n                z_norm_pt = np.clip((q[:, 2] - z_min) / z_span, 0.0, 1.0)\n                u_norm_pt = np.abs(2.0 * (u_img / max(float(pw - 1), 1.0)) - 1.0)\n                v_norm_pt = v_img / max(float(ph - 1), 1.0)\n                end_fascia = (\n                    front_shell\n                    & (z_norm_pt > 0.70)\n                    & (v_norm_pt > 0.20)\n                    & (u_norm_pt < 0.92)\n                    & (facing_pos > 0.10)\n                )\n                facing_eff = np.where(end_fascia, np.maximum(facing_eff, 0.92), facing_eff).astype(\'f\')\n                fascia_boost = np.where(end_fascia, 3.5, 1.0).astype(\'f\')\n            else:\n                fascia_boost = np.float32(1.0)\n\n            reach_w = _depth_reach(base_label, q[:, 2], z_min, z_span, u_img, v_img, pw, ph)\n            rgba = _sample_rgba_bilinear(pic_f, u_img, v_img)\n            alpha_ok = np.clip((rgba[:, 3] - 128.0) / 100.0, 0.0, 1.0).astype(\'f\')\n\n            w_cam = (\n                (facing_eff ** 6)\n                * fascia_boost\n                * vis_factor\n                * alpha_ok\n                * border_map[py, px]\n                * jump_map[py, px]\n                * reach_w\n                * score_pow\n                * view_priority\n            )\n            conf_cam = np.where(\n                front_shell & (alpha_ok > 0.25),\n                np.clip(facing_eff / 0.08, 0.35, 1.0) * alpha_ok * conf_scale,\n                0.0,\n            ).astype(\'f\')\n            weighted[yy, xx] += w_cam[:, None] * rgba[:, :3]\n            weights[yy, xx] += w_cam\n            max_conf[yy, xx] = np.maximum(max_conf[yy, xx], conf_cam)\n\n    valid = owner >= 0\n    mix = np.where(weights > 1e-8, np.clip(max_conf / 0.65, 0.0, 1.0), 0.0).astype(\'f\')\n    photo_rgb = weighted / np.maximum(weights[:, :, None], 1e-10)\n    fused = atlas.astype(np.float32) * (1.0 - mix[:, :, None]) + photo_rgb * mix[:, :, None]\n    _inpaint_unobserved(fused, mix, vertices, faces, owner, bary, valid, model_diag)\n\n    mat.baseColorTexture = Image.fromarray(np.uint8(np.clip(fused, 0, 255)))\n    for item,label in zip(report,supplied_labels):\n        item[\'supplied_label\'] = label\n    return dict(\n        reference_registration=report,\n        photo_texture_fraction=round(float(np.count_nonzero(mix[valid] > .5) / max(1, int(valid.sum()))), 4),\n    )\n\n\ndef unlit(document):\n    document.setdefault(\'extensionsUsed\',[]).append(\'KHR_materials_unlit\')\n    for material in document[\'materials\']:\n        material.setdefault(\'extensions\',{})[\'KHR_materials_unlit\'] = {}\n'

if 'studio_proc' in globals() and studio_proc.poll() is None:
    raise RuntimeError('The service is already running. Run the final cell to copy its URL, or restart the runtime for changed code.')
STUDIO_KEY=secrets.token_urlsafe(32)
path=pathlib.Path('/content/TRELLIS/form_service.py')
path.write_text(service_source)
path.with_name('photo_finish.py').write_text(photo_source)
env=os.environ.copy()
env.update(STUDIO_KEY=STUDIO_KEY, PATH='/content/trellis-env/bin:'+env['PATH'], CUDA_HOME='/content/trellis-env', CC='/content/trellis-env/bin/x86_64-conda-linux-gnu-cc', CXX='/content/trellis-env/bin/x86_64-conda-linux-gnu-c++', ATTN_BACKEND='xformers', SPCONV_ALGO='native', TORCH_CUDA_ARCH_LIST='8.0', MAX_JOBS='4')
env['CPATH']=':'.join(['/content/trellis-env/targets/x86_64-linux/include']+[str(p) for p in pathlib.Path('/content/trellis-env/lib/python3.10/site-packages/nvidia').glob('*/include')])
env['LIBRARY_PATH']='/content/trellis-env/targets/x86_64-linux/lib'
log=open('/content/form-service.log','w')
studio_proc=subprocess.Popen(['/content/trellis-env/bin/python','-u',str(path)],cwd='/content/TRELLIS',env=env,stdout=log,stderr=subprocess.STDOUT)
for attempt in range(600):
    if studio_proc.poll() is not None:
        raise RuntimeError(pathlib.Path('/content/form-service.log').read_text()[-6000:])
    try:
        request=urllib.request.Request('http://127.0.0.1:7860/health',headers={'Authorization':'Bearer '+STUDIO_KEY})
        with urllib.request.urlopen(request,timeout=2) as response:
            if response.status==200:break
    except Exception:time.sleep(2)
else:raise TimeoutError('Loading is taking longer than expected. Inspect /content/form-service.log before restarting.')
print('TRELLIS ready. Run the final cell to get your connection URL.')


In [ ]:
# @title 3. Copy this connection URL into localhost
import pathlib, subprocess, urllib.request, re, time
if 'studio_proc' not in globals() or studio_proc.poll() is not None:
    raise RuntimeError('Run the service cell first.')
if 'tunnel_proc' not in globals() or tunnel_proc.poll() is not None:
    binary=pathlib.Path('/content/cloudflared')
    if not binary.exists():
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',binary)
        binary.chmod(0o755)
    tunnel_log=open('/content/form-tunnel.log','w')
    tunnel_proc=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
for attempt in range(90):
    if tunnel_proc.poll() is not None:raise RuntimeError('Tunnel stopped; inspect /content/form-tunnel.log')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',pathlib.Path('/content/form-tunnel.log').read_text())
    if match:break
    time.sleep(1)
else:raise TimeoutError('Tunnel URL not ready; rerun this cell.')
print('Paste this entire URL into MeshPort (keep it private):')
print(match.group(0)+'/#key='+STUDIO_KEY)
print('Keep this Colab runtime connected. Reconnecting a new runtime creates a new URL.')
